In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape
import numpy as np
from pathlib import Path
import os
import sys

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
PARENT_DIRECTORY = os.path.dirname(CURRENT_DIRECTORY)
print(PARENT_DIRECTORY)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src


In [3]:
BASE_DATASET_PATH = Path(PARENT_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [4]:
QGIS_PATH = '/Applications/QGIS-LTR.app/Contents/Resources/python' 

# Add the QGIS Python path to the system path
sys.path.append(QGIS_PATH)
# Add path to plugins directory (usually needed for 'processing' module)
sys.path.append(os.path.join(QGIS_PATH, 'plugins'))

# QgsApplication.setPrefixPath() tells QGIS where to look for its core C++ libraries (plugins, resources, etc.).
# The first argument is the QGIS installation prefix (usually the folder containing 'bin', 'apps', etc.)
# For your Mac example, we use the path leading up to 'Resources'.
qgis_app_prefix = '/Applications/QGIS-LTR.app/Contents/Resources' # Adjust as needed

print(f"1. Setting QGIS prefix path to: {qgis_app_prefix}")
from qgis.core import QgsApplication
QgsApplication.setPrefixPath(qgis_app_prefix, True)

# Create an instance of QgsApplication. The second argument (False) means no GUI.
# This prepares the environment for non-GUI (standalone) scripting.
qgs = QgsApplication([], False) 

# Load Providers and registries (REQUIRED)
print("2. Initializing QGIS application...")
qgs.initQgis() 
print("QGIS application initialized successfully.")

1. Setting QGIS prefix path to: /Applications/QGIS-LTR.app/Contents/Resources
2. Initializing QGIS application...
QGIS application initialized successfully.


In [5]:
# Import QGIS Libraries and Processing
try:
    # Now that QGIS is initialized, we can safely import these modules
    from qgis.analysis import QgsNativeAlgorithms
    import processing
    from processing.core.Processing import Processing
    from qgis.core import QgsProject, QgsProcessingFeedback, QgsVectorLayer, QgsCoordinateReferenceSystem
    from qgis.PyQt.QtCore import QVariant 
except ImportError as e:
    print(f"\n--- FATAL ERROR: MODULE IMPORT FAILED AFTER INITIALIZATION ---")
    print(f"Please double-check the QGIS_PATH and qgis_app_prefix variables.")
    print(f"Original error: {e}")
    sys.exit(1)

In [6]:
# initialise the Processing Framework
Processing.initialize()
# Add the QGIS native algorithms provider
QgsApplication.processingRegistry().addProvider(QgsNativeAlgorithms())

Logged warning: Duplicate provider native registered


False

In [7]:
# reading singapore 2021 multipolygon map data
singapore_2021_filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_2021_multipolygons.gpkg"
singapore_2021_gpd = gpd.read_file(singapore_2021_filepath)
singapore_2021_gpd.crs

<Projected CRS: EPSG:3414>
Name: SVY21 / Singapore TM
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Singapore - onshore and offshore.
- bounds: (103.59, 1.13, 104.07, 1.47)
Coordinate Operation:
- name: Singapore Transverse Mercator
- method: Transverse Mercator
Datum: SVY21
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [8]:
# add a manual_tags column
singapore_2021_gpd["manual_tags"] = None
singapore_2021_gpd.columns

Index(['osm_id', 'osm_way_id', 'name', 'type', 'aeroway', 'amenity',
       'admin_level', 'barrier', 'boundary', 'building', 'craft', 'geological',
       'historic', 'land_area', 'landuse', 'leisure', 'man_made', 'military',
       'natural', 'office', 'place', 'shop', 'sport', 'tourism', 'other_tags',
       'geometry', 'manual_tags'],
      dtype='object')

In [9]:
# commercial column is not descriptive enough. 
# rows where commercial != null AND other_tags contains HDB will be tagged with street_shops
# rows where commerical != nulul AND shop == "mall" will be tagged with mall
singapore_landuse = singapore_2021_gpd[singapore_2021_gpd["landuse"] == "commercial"].copy()

hdb_condition = singapore_landuse["other_tags"].str.contains("HDB", na = False)
mall_condition = singapore_landuse["shop"] == "mall"

singapore_landuse[["other_tags", "shop"]] = singapore_landuse[["other_tags", "shop"]].astype(str)

singapore_landuse.loc[hdb_condition, "manual_tags"] = "street_shops"
singapore_landuse.loc[mall_condition, "manual_tags"] = "mall"

singapore_2021_gpd.update(singapore_landuse[["manual_tags"]])

file_path = str(BASE_DATASET_PATH / "geospatial_data/2021_geospatial/commercial_polygons.xlsx")
singapore_landuse.to_excel(file_path)

singapore_2021_gpd["manual_tags"].unique()

array([None, 'mall', 'street_shops'], dtype=object)

In [10]:
# collating the changes and saving to another gpkg file
filepath = str(BASE_DATASET_PATH / "geospatial_data/2021_geospatial/multipolygons_edited_2021.gpkg")
singapore_2021_gpd.to_file(filepath)

In [11]:
singapore_2021_lines_filepath = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_2021_lines.gpkg"
singapore_2021_lines = gpd.read_file(singapore_2021_lines_filepath)
singapore_2021_lines.crs

<Projected CRS: EPSG:3414>
Name: SVY21 / Singapore TM
Axis Info [cartesian]:
- N[north]: Northing (metre)
- E[east]: Easting (metre)
Area of Use:
- name: Singapore - onshore and offshore.
- bounds: (103.59, 1.13, 104.07, 1.47)
Coordinate Operation:
- name: Singapore Transverse Mercator
- method: Transverse Mercator
Datum: SVY21
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [12]:
# https://wiki.openstreetmap.org/wiki/WikiProject_Singapore#Roads
print(f"{singapore_2021_lines.columns} \n")
print(singapore_2021_lines['highway'].unique())

Index(['osm_id', 'name', 'highway', 'waterway', 'aerialway', 'barrier',
       'man_made', 'z_order', 'other_tags', 'geometry'],
      dtype='object') 

['primary' 'residential' 'tertiary' 'footway' 'service' 'secondary'
 'motorway' 'motorway_link' 'trunk' None 'trunk_link' 'primary_link'
 'secondary_link' 'living_street' 'construction' 'pedestrian'
 'unclassified' 'proposed' 'steps' 'track' 'cycleway' 'path'
 'tertiary_link' 'bridleway' 'raceway' 'corridor' 'elevator' 'rest_area']


#### Based on the wiki, I have decided to include motorway, trunk, primary, secondary as the major roads in singapore to include

In [13]:
major_roads = [
    'motorway', 
    'trunk', 
    'primary', 
    'secondary'
]

singapore_major_roads = singapore_2021_lines[
    singapore_2021_lines['highway'].isin(major_roads)
].copy()

file_path = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_major_roads_2021.gpkg"
singapore_major_roads.to_file(file_path)

#### Combine the lines of major roads into one singular line
There will be feature loss but the aim was to create a polygon of the major roads. There wasnt a need to preserve the road's features.

In [14]:
combined_major_roads = singapore_major_roads.dissolve()
combined_major_roads.columns
# singapore_2021_gpd.columns

Index(['geometry', 'osm_id', 'name', 'highway', 'waterway', 'aerialway',
       'barrier', 'man_made', 'z_order', 'other_tags'],
      dtype='object')

In [28]:
drop_columns = ["osm_id", "name", "highway", "waterway", "aerialway", "barrier", "man_made", "z_order", "other_tags"]
combined_major_roads = combined_major_roads.drop(drop_columns, axis = 1, errors = 'ignore')
combined_major_roads["major_roads"] = "yes"
print(combined_major_roads)
file_path = BASE_DATASET_PATH / "geospatial_data/2021_geospatial/singapore_combined_major_roads_2021.gpkg"
# # save to gpkg file for processing using qgis
combined_major_roads.to_file(file_path)

                                            geometry major_roads
0  MULTILINESTRING ((27637.517 32038.397, 27643.1...         yes


In [15]:
# transforming the lines into polygon using the buffer function in qgis
processing.run("native:buffer", {'INPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/singapore_combined_major_roads_2021.gpkg|layername=singapore_combined_major_roads_2021','DISTANCE':5,'SEGMENTS':5,'END_CAP_STYLE':0,'JOIN_STYLE':0,'MITER_LIMIT':2,'DISSOLVE':True,'SEPARATE_DISJOINT':False,'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/2021_combined_major_roads_polygon.gpkg'})

{'OUTPUT': '/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/2021_combined_major_roads_polygon.gpkg'}

### Creating a grid over Singapore using qgis
The grid size will be a hectare large ($100m \times 100m$)

The extents of the grid was determined by looking at Google maps to find the boundaries of Singapore. And converting to 

In [ ]:
processing.run("native:creategrid", {'TYPE':2,'EXTENT':'2666.773500000,56466.773500000,15657.950600000,50257.950600000 [EPSG:3414]','HSPACING':100,'VSPACING':100,'HOVERLAY':0,'VOVERLAY':0,'CRS':QgsCoordinateReferenceSystem('EPSG:3414'),'OUTPUT':'/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/geospatial_data/2021_geospatial/hectare_grid.gpkg'})

```
ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  hectare_grid.gpkg \
  -nln hectare_grid \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  hectare_grid

ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  hectare_grids.gpkg \
  -nln buildings_3414 \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  buildings_3414

ogr2ogr -f PostgreSQL \
  PG:"dbname=postgis user=yitong" \
  hectare_grids.gpkg \
  -nln landuse_3414 \
  -lco GEOMETRY_NAME=geom \
  -lco FID=id \
  -nlt PROMOTE_TO_MULTI \
  landuse_3414
```